# Setup environment for Zotero API

In [14]:
# conda install pyzotero
import sys
!conda install --yes -c conda-forge --prefix {sys.prefix} pyzotero python-dateutil

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 4.12.0
  latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.



# Connect to Zotero API

In [37]:
from pyzotero import zotero

library_id = '6004137'
library_type = 'group'
api_key = 'ZW7JhiiB1JGF0RBZV7frgBgm'

zot = zotero.Zotero(library_id, library_type, api_key)
# check if the connection is successful
sample_items = zot.top(limit=5)
# we've retrieved the latest five top-level items in our library
# we can print each item's item type and ID
for item in sample_items:
    print('Item Type: %s | Key: %s' % (item['data']['itemType'], item['data']['key']))

Item Type: journalArticle | Key: YSVB4T4A
Item Type: journalArticle | Key: YMK2MXUE
Item Type: journalArticle | Key: FKBGWXQ7
Item Type: journalArticle | Key: KAQNAIR3
Item Type: bookSection | Key: FHSSHWMX


In [38]:
all_items = zot.everything(zot.items())

In [33]:
print(len(all_items))

6237


# Count the number of occurence of papers in different query results (each query is a collection in this case)

## First explore collection to see duplicates in titles

In [5]:
# Find all items with a duplicate title
from collections import defaultdict

title_counts = defaultdict(int)
for item in all_items:
    # Only consider items with itemType ≠ 'note' or 'attachment'
    if item['data']['itemType'] in ['note', 'attachment']:
        continue
    title = item['data']['title']
    title_counts[title] += 1
# Print titles that have duplicates
for title, count in title_counts.items():
    if count > 1:
        print(f"Title: {title} | Count: {count}")



Title: Introduction | Count: 2
Title: Machine translation with a stochastic grammatical channel | Count: 2
Title: Learning Hierarchical Lexical Hyponymy | Count: 2
Title: Parsing Turkish using the lexical functional grammar formalism | Count: 2
Title: Human Language Technology. Challenges for Computer Science and Linguistics | Count: 3
Title: Natural Language Generation | Count: 2
Title: Pangloss: a knowledge-based machine assisted translation research project | Count: 2
Title: Ontologies | Count: 3
Title: Neural Machine Translation and Linked Data | Count: 2
Title: Ontology Localization | Count: 2


## Updating all items with the number of collections they belong to as a tag


In [14]:
# For each item, count the number of Collections it belongs in add it as a tag into the entry

def update_collection_count_tags():
    """Update each item's tags with the number of collections it belongs to."""
    hold = True
    for item in all_items:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        collections = item.get('data', {}).get('collections', [])
        # filter out collections beginning with '@'
        collections = [c for c in collections if not c.startswith('@')]
        tag_value = f"#collections:{len(collections)}"
        print("Updating item with key:", item['data']['key'], "with tag:", tag_value)
        # Add the tag to the item if it doesn't already exist
        if tag_value not in item['data'].get('tags', []):
            zot.add_tags(item, tag_value)


# update_collection_count_tags()


Updating item with key: KLH85E8Z with tag: #collections:2
Updating item with key: K6534KAR with tag: #collections:4
Updating item with key: M4BH9AL7 with tag: #collections:2
Updating item with key: 2GFY7I8P with tag: #collections:2
Updating item with key: ML6FXQV8 with tag: #collections:2
Updating item with key: ERBNP6MG with tag: #collections:2
Updating item with key: 9SI2FX7X with tag: #collections:2
Updating item with key: 6Q2KM42N with tag: #collections:2
Updating item with key: LAHWYEQ6 with tag: #collections:2
Updating item with key: LSATUK73 with tag: #collections:2
Updating item with key: VC5PJY8L with tag: #collections:3
Updating item with key: 8H5E79RW with tag: #collections:2
Updating item with key: SU99TFCR with tag: #collections:4
Updating item with key: AVEVMJT9 with tag: #collections:4
Updating item with key: HJVNNUCZ with tag: #collections:2
Updating item with key: FJ2ZWANP with tag: #collections:2
Updating item with key: 56DB6EXK with tag: #collections:2
Updating item 

ReadTimeout: The read operation timed out

# Assign papers to the different reviewers, keeping only items from 2018 and later


In [30]:
from dateutil import parser

reviewers = [
    "@Mike",
    "@Gilles",
    "@Ana",
    "@Dagmar",
    "@Ciprian",
    "@Elena",
    "@Buket"]

## Define a function that takes a date string and returns the year
def get_year_from_date(date_str):
    """Extracts the year from a zotero date string.
    examples:
    '2025' --> 2025
    '2025-01-01' --> 2025
    '2025-01' --> 2025
    '2025 DEC' --> 2025
    '06/2024' --> 2024
    '3 July 2019' --> 2019"""
    if not date_str:
        return None
    try:
        year_candidate = date_str.split('-')[0]
        # Check if the first part is a valid year
        if year_candidate.isdigit() and len(year_candidate) == 4:
            return int(year_candidate)
        year_candidate = date_str.split(' ')[0]
        # Check if the first part is a valid year in a different format
        if year_candidate.isdigit() and len(year_candidate) == 4:
            return int(year_candidate)
        # If the date is in a different format, try to parse it
        parsed_date = parser.parse(date_str, fuzzy=True)
        return parsed_date.year
    except ValueError:
        return None

def assign_items_to_reviewers():
    """Assign items from 2018 and later to reviewers in a round-robin fashion."""
    global reviewers
    # Create a dictionary to hold the items for each reviewer
    reviewer_items = {reviewer: [] for reviewer in reviewers}
    # Filter items by year and assign to reviewers
    item_number = 0
    for item in all_items:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        # Check if the item has a date and if it is 2018 or later
        date = item['data'].get('date', '')
        if date and get_year_from_date(date) >= 2018:
            item_number += 1
            # Assign the item to a reviewer in a round-robin fashion
            reviewer = reviewers[item_number % len(reviewers)]
            reviewer_items[reviewer].append(item)

    # Print the number of items assigned to each reviewer
    for reviewer, ritems in reviewer_items.items():
        print(f"{reviewer}: {len(ritems)} items")

    # Then put the items into the corresponding collection using Zotero API
    for reviewer, ritems in reviewer_items.items():
        # Create a collection for the reviewer if it doesn't exist
        collection_name = f"{reviewer}"
        collections = zot.collections()
        collection = next((c for c in collections if c['data']['name'] == collection_name), None)
        if not collection:
            collection = zot.create_collection([{'name': collection_name}])
            if collection['success']['0']:
                collection_id = collection['success']['0']
            else:
                print(f"Failed to create collection {collection_name}")
                continue
        else:
            collection_id = collection['data']['key']

        # Add items to the collection
        for item in ritems:
            zot.addto_collection(collection_id, item)
            print(f"Added item {item['data']['title']} to collection {collection_name}")


@Mike: 179 items
@Gilles: 180 items
@Ana: 180 items
@Dagmar: 179 items
@Ciprian: 179 items
@Elena: 179 items
@Buket: 179 items


In [31]:
# assign_items_to_reviewers()

Added item Towards language-agnostic alignment of product titles and descriptions: a neural approach to collection @Mike
Added item Retrieve-and-Fill for Scenario-based Task-Oriented Semantic Parsing to collection @Mike
Added item ANALYSIS OF TEXT AUGMENTATION ALGORITHMS IN ARTIFICIAL LANGUAGE MACHINE TRANSLATION SYSTEMS to collection @Mike
Added item BomJi at SemEval-2018 Task 10: Combining Vector-, Pattern- and Graph-based Information to Identify Discriminative Attributes to collection @Mike
Added item Unifying dimensions in coherence relations: How various annotation frameworks are related to collection @Mike
Added item Why Can Computers Understand Natural Language? to collection @Mike
Added item Recent Progress, Emerging Techniques, and Future Research Prospects of Bangla Machine Translation: A Systematic Review to collection @Mike
Added item Experiment on English-Thai Machine Translation via Text Understanding Based on Mental Image Directed Semantic Theory to collection @Mike
Adde

# Add a note to each item in the collection


In [18]:
note_html = """<div data-schema-version="9"><p>[T3.3 REVIEW]</p>
<ul>
<li>
Does it talk about KG AND MT (yes/no):
</li>
<li>
Direction of influence (KG_for_MT/MT_for_KG/INDIRECT):
</li>
<li>
Relevant (include it in next pass or just forget it, yes/no/maybe):
</li>
<li>
—
</li>
<li>
Kind of KG (Ontology/Language/Multilingual/…):
</li>
<li>
Kind of MT (RBMT/EBMT/SMT/NMT/LLM/OTHER):
</li>
</ul>
<p>Remark (if applicable): </p>
</div>
"""


In [29]:
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [28]:


def add_note_templates():
    """Add a note template to each item in the library."""
    notes = []
    for item in all_items:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        # Create a note for the item
        note = {
            'itemType': 'note',
            'note': note_html,
            'parentItem': item['data']['key'],
        }
        notes.append(note)

    for chunk in chunk_list(notes, 40):
        zot.create_items(chunk)
        print(f"Created {len(chunk)} notes")


# add_note_templates()

# Explore the current items and add tags to follow reviewing work advancement

In [39]:
def get_item_by_key(key):
    global all_items
    """Return the item dict from all_items whose data.key == key, or None if not found."""
    return next((it for it in all_items if it.get('key') == key), None)


def tag_review_status():
    """Tag items associated to a review note that is equals to the notes_html."""
    count = 0
    dirty_items = []
    for item in all_items:
        # Only consider notes
        if item['data']['itemType'] in ['note']:
            # filter notes that have nothing to do with GOBLIN review
            if '[T3.3 REVIEW]' not in item['data']['note']:
                continue
            note_text = item['data']['note']
            parent_item_key = item['data'].get('parentItem')
            parent_item = get_item_by_key(parent_item_key)
            if not parent_item:
                print(f"Note {item['key']} has no parent item")
                continue
            parent_item_tags = [t["tag"] for t in parent_item['data'].get('tags', [])]
            parent_item_is_dirty = False
            if note_text.strip() != note_html.strip():
                # Note has been modified, tag the item as '#reviewed'
                count += 1
                # print(f"Item {parent_item_key} has note {note_text}")
                if '#reviewed' not in parent_item_tags:
                    parent_item_tags.append('#reviewed')
                    parent_item_is_dirty = True
                if '#to_review' in parent_item_tags:
                    parent_item_tags.remove('#to_review')
                    parent_item_is_dirty = True
            else:
                # Note is unchanged, tag the item as '#to_review'
                if '#to_review' not in parent_item_tags:
                    parent_item_tags.append('#to_review')
                    parent_item_is_dirty = True
            if parent_item_is_dirty:
                parent_item_clone = parent_item.copy()
                parent_item_clone["data"] = {}
                parent_item_clone["data"]["tags"] = []
                for tag in parent_item_tags:
                    parent_item_clone["data"]["tags"].append({"tag": f"{tag}"})
                zot.update_item(parent_item_clone)
                dirty_items.append(parent_item_clone)
    print(f"{count} items with review notes found.")
    print(f"{len(dirty_items)} items to update.")
    return
    for chunk in chunk_list(dirty_items, 40):
        zot.check_items(chunk)
        zot.update_item(chunk)

tag_review_status()

75 items with review notes found.
6 items to update.


# Debug stuff, just ignore it

In [24]:
zot.collections()

[{'key': '7QNUW22V',
  'version': 5823,
  'library': {'type': 'group',
   'id': 6004137,
   'name': 'KG_and_MT_GOBLIN_T3.3',
   'links': {'alternate': {'href': 'https://www.zotero.org/groups/6004137',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'https://api.zotero.org/groups/6004137/collections/7QNUW22V',
    'type': 'application/json'},
   'alternate': {'href': 'https://www.zotero.org/groups/6004137/collections/7QNUW22V',
    'type': 'text/html'}},
  'meta': {'numCollections': 0, 'numItems': 0},
  'data': {'key': '7QNUW22V',
   'version': 5823,
   'name': '@Mike',
   'parentCollection': False,
   'relations': {}}},
 {'key': '86QPMQSU',
  'version': 4544,
  'library': {'type': 'group',
   'id': 6004137,
   'name': 'KG_and_MT_GOBLIN_T3.3',
   'links': {'alternate': {'href': 'https://www.zotero.org/groups/6004137',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'https://api.zotero.org/groups/6004137/collections/86QPMQSU',
    'type': 'application/json'},
   'alte

In [11]:
zot.create_collection([{'name': '@Testing Collection'}])

{'successful': {'0': {'key': '9GRRS9DV',
   'version': 4552,
   'library': {'type': 'group',
    'id': 6004137,
    'name': 'KG_and_MT_GOBLIN_T3.3',
    'links': {'alternate': {'href': 'https://www.zotero.org/groups/6004137',
      'type': 'text/html'}}},
   'links': {'self': {'href': 'https://api.zotero.org/groups/6004137/collections/9GRRS9DV',
     'type': 'application/json'},
    'alternate': {'href': 'https://www.zotero.org/groups/6004137/collections/9GRRS9DV',
     'type': 'text/html'}},
   'meta': {'numCollections': 0, 'numItems': 0},
   'data': {'key': '9GRRS9DV',
    'version': 4552,
    'name': '@Mike',
    'parentCollection': False,
    'relations': {}}}},
 'success': {'0': '9GRRS9DV'},
 'unchanged': {},
 'failed': {}}

In [6]:
# retrieve the first note in items and display it
notes = [item for item in all_items if item['data']['itemType'] == 'note']

In [20]:
print(zot.item_types())

[{'itemType': 'artwork', 'localized': 'Artwork'}, {'itemType': 'audioRecording', 'localized': 'Audio Recording'}, {'itemType': 'bill', 'localized': 'Bill'}, {'itemType': 'blogPost', 'localized': 'Blog Post'}, {'itemType': 'book', 'localized': 'Book'}, {'itemType': 'bookSection', 'localized': 'Book Section'}, {'itemType': 'case', 'localized': 'Case'}, {'itemType': 'conferencePaper', 'localized': 'Conference Paper'}, {'itemType': 'dataset', 'localized': 'Dataset'}, {'itemType': 'dictionaryEntry', 'localized': 'Dictionary Entry'}, {'itemType': 'document', 'localized': 'Document'}, {'itemType': 'email', 'localized': 'E-mail'}, {'itemType': 'encyclopediaArticle', 'localized': 'Encyclopedia Article'}, {'itemType': 'film', 'localized': 'Film'}, {'itemType': 'forumPost', 'localized': 'Forum Post'}, {'itemType': 'hearing', 'localized': 'Hearing'}, {'itemType': 'instantMessage', 'localized': 'Instant Message'}, {'itemType': 'interview', 'localized': 'Interview'}, {'itemType': 'journalArticle', '